# CUDA development on a free GPU

This machine has no NVIDIA GPU, so kernels run here in Colab instead.

## Before you run anything

**Runtime -> Change runtime type -> T4 GPU -> Save.**

Without this you get a CPU-only runtime and every cell below fails. The free tier
normally hands out a Tesla T4 (compute capability 7.5).

## 1. Confirm the GPU and toolkit

`nvidia-smi` shows the device and the **driver's** CUDA version; `nvcc --version`
shows the **toolkit** version. These two often differ on Colab, which is fine --
`scripts/build.sh` compiles for the GPU's real architecture specifically so a
newer toolkit never has to JIT PTX through an older driver.

In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!nvcc --version

## 2. Get the code

Edit `REPO_URL` below to point at your GitHub repo. This cell is idempotent --
it clones on the first run and pulls on every reconnect afterwards.

In [ ]:
import os

REPO_URL = "https://github.com/edutms/cuda_development_learning.git"
WORKDIR  = "/content/cuda_development_learning"

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    os.chdir(WORKDIR)
    !git pull --ff-only
else:
    !git clone {REPO_URL} {WORKDIR}
    os.chdir(WORKDIR)

print("\nworking directory:", os.getcwd())
!ls

## 3. Example 01 - hello kernel

Smallest possible launch: 2 blocks of 4 threads, each printing its own indices.
If this prints per-thread lines, the full local -> git -> Colab -> GPU loop works.

In [ ]:
!bash scripts/build.sh src/01_hello/hello.cu && ./build/hello

## 4. Example 02 - vector add

The complete host/device cycle over 1M elements, with CUDA-event timing.
Expect `PASSED` and an effective bandwidth figure. A T4 has ~320 GB/s of
theoretical bandwidth, and this kernel is purely memory-bound, so the number
you get is a good sanity check on whether the GPU is behaving.

In [ ]:
!bash scripts/build.sh src/02_vector_add/vector_add.cu && ./build/vector_add

## 5. Scratch kernels

For quick throwaway experiments you do not want to commit, write the file
straight from a cell with `%%writefile` and build it the same way.

In [ ]:
%%writefile /content/scratch.cu
#include <cstdio>

__global__ void scratch_kernel() {
    printf("thread %d reporting\n", threadIdx.x);
}

int main() {
    scratch_kernel<<<1, 8>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -std=c++17 /content/scratch.cu -o /content/scratch && /content/scratch

## Keeping your work

**`/content` is wiped when the runtime disconnects.** Git is the only thing that
persists. Anything worth keeping goes back to the repo:

```
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add -A && git commit -m "..." && git push
```

Pushing needs a GitHub personal access token rather than a password. The lower-friction
habit is to edit locally, `make check` to catch compile errors without a GPU, push from
your machine, and re-run cell 2 here to pull.